<a href="https://colab.research.google.com/github/sundarrajan-mugunthan/Fashion_recommendation/blob/main/Fashion_Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy faiss-cpu sentence-transformers transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.3 MB/s eta 0:00:00


In [5]:
!kaggle datasets download -d roopacalistus/superstore
!unzip superstore.zip
!ls

Dataset URL: https://www.kaggle.com/datasets/roopacalistus/superstore
License(s): GNU Lesser General Public License 3.0
100% 164k/164k [00:00<00:00, 98.7MB/s]

Archive:  superstore.zip
  inflating: SampleSuperstore.csv    
sample_data  SampleSuperstore.csv  superstore.zip


In [8]:
!mv "SampleSuperstore.csv" Superstore.csv

In [9]:
import pandas as pd
df = pd.read_csv('Superstore.csv')
df.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [10]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 71.7 MB/s eta 0:00:00


In [11]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import random

st.set_page_config(page_title="Fashion Recommender", layout="wide")

st.title("👕 AI Fashion Recommender & Outfit Builder")

# -----------------------------
# Load Dataset
# -----------------------------
@st.cache_data
def load_data():
    df = pd.read_csv('Superstore.csv')

    df_fashion = pd.DataFrame()
    df_fashion['product_name'] = df['Product Name']
    df_fashion['category'] = df['Category']
    df_fashion['sub_category'] = df['Sub-Category']
    df_fashion['price'] = df['Sales']

    colors = ["Black", "White", "Blue", "Beige", "Green", "Red"]
    styles = ["Casual", "Formal", "Sport", "Minimal", "Streetwear"]

    df_fashion['color'] = [random.choice(colors) for _ in range(len(df_fashion))]
    df_fashion['style'] = [random.choice(styles) for _ in range(len(df_fashion))]

    df_fashion = df_fashion.dropna().sample(3000, random_state=42)

    def create_text(row):
        return f"{row['product_name']} {row['category']} {row['sub_category']} {row['color']} {row['style']}"

    df_fashion['combined_text'] = df_fashion.apply(create_text, axis=1)

    return df_fashion

df_fashion = load_data()

# -----------------------------
# Load Models
# -----------------------------
@st.cache_resource
def load_models():
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    generator = pipeline(
        "text2text-generation",
        model="google/flan-t5-base",
        device=-1
    )

    return embed_model, generator

embed_model, generator = load_models()

# -----------------------------
# FAISS Index
# -----------------------------
@st.cache_resource
def create_index(df):
    embeddings = embed_model.encode(df['combined_text'].tolist())
    embeddings = np.array(embeddings)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

index = create_index(df_fashion)

# -----------------------------
# Functions
# -----------------------------
def search(query, k=10):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k)
    return df_fashion.iloc[I[0]]

def map_category(row):
    text = row['sub_category'].lower()
    if "shirt" in text or "top" in text:
        return "top"
    elif "pants" in text or "jeans" in text:
        return "bottom"
    elif "shoe" in text:
        return "shoes"
    return "other"

df_fashion['main_category'] = df_fashion.apply(map_category, axis=1)

def build_outfit(results):
    outfit = {}
    for _, row in results.iterrows():
        cat = row['main_category']
        if cat not in outfit:
            outfit[cat] = row
    return outfit

def stylist(query, results):
    items = "\n".join([
        f"{i+1}. {row['product_name']} ({row['color']}, {row['style']})"
        for i, row in results.iterrows()
    ])

    prompt = f"""
    Create outfit for: {query}

    Items:
    {items}

    Output:
    Top:
    Bottom:
    Shoes:
    Reason:
    """

    response = generator(prompt, max_length=150)[0]['generated_text']
    return response

# -----------------------------
# UI
# -----------------------------
query = st.text_input("Describe your outfit", "casual summer outfit for office")

if st.button("Generate Outfit"):
    results = search(query)
    outfit = build_outfit(results)
    explanation = stylist(query, results)

    st.subheader("🔍 Recommended Items")
    st.dataframe(results[['product_name', 'category', 'color', 'style']])

    st.subheader("👕 Outfit")
    for k, v in outfit.items():
        st.write(f"**{k.upper()}** → {v['product_name']} ({v['color']})")

    st.subheader("🧠 AI Stylist")
    st.write(explanation)

Writing app.py


In [12]:
from pyngrok import ngrok

!streamlit run app.py &

public_url = ngrok.connect(8501)
print(public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.115.65.169:8501

  Stopping...


ERROR:pyngrok.process.ngrok:t=2026-03-28T20:16:32+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-28T20:16:32+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.